# 04 Inventory Performance Analysis — 開發日誌

**資料來源：**
- Kaggle: kaggle.com/bhanupratapbiswas/inventory-analysis-case-study
- PwC 是 PricewaterhouseCoopers 的縮寫，全球四大會計師事務所（Big Four）之一
- 數據集的業務邏輯設計符合真實審計和財務分析標準，欄位設計反映的是真實企業的採購流程


---
## 📅 2026-04-14 — Phase 1：Raw Data 載入

### ✅ 完成事項
- 建立 `raw` / `staging` / `marts` 三層 Schema
- 執行 `sql/00_schema_setup.sql` 成功建立所有 raw tables
- 完成 `scripts/01_load_raw.py` Python Loader 腳本

---

### 🪲 踩坑紀錄 1：`00_schema_setup.sql` 出現 NOTICE 訊息

**現象：**
```
NOTICE: table "raw_sales" does not exist, skipping
Successfully run. Total query runtime: 113 msec.
1 rows affected.
```

**原因：**
- `NOTICE` 只是 PostgreSQL 的溫馨提示，不是錯誤（ERROR）
- SQL 中使用了 `DROP TABLE IF EXISTS`，第一次執行時找不到資料表，PostgreSQL 會提示 skipping
- `1 rows affected` 是最後一行 `SELECT 'Schema setup complete ✅'` 回傳了 1 行結果

**結論：** ✅ 完全正常，可放心繼續執行

---

### 🪲 踩坑紀錄 2：執行 Python 腳本時彈出新視窗、自動關閉

**現象：**
- 在 VSCode PowerShell 執行 `01_load_raw.py` 時，自動跳出一個新的黑色 Console 視窗
- 視窗跑完後自動關閉，看不到輸出結果

**原因：**
- Windows 環境下，某些情況下 Python 腳本會以獨立視窗模式啟動
- 程式執行結果無法保留在 VSCode 終端機中

**解決方法：**
- 直接在 VSCode 的終端機（PowerShell）輸入指令執行，而不是按 Run 按鈕
- 加上 `-u` 參數（Unbuffered）：
```bash
python -u scripts/01_load_raw.py
```

---

### 🪲 踩坑紀錄 3：`KeyboardInterrupt` 錯誤反覆出現

**現象：**
```
Traceback (most recent call last):
  File "scripts/01_load_raw.py", line 12, in <module>
    from sqlalchemy import create_engine, text
  ...
KeyboardInterrupt
```

**原因：**
- 這不是程式本身的 Bug！
- Python 在 Windows 上第一次載入 `pandas`、`sqlalchemy` 等大型套件時，需要花 **30 秒至 1 分鐘** 從硬碟讀取套件檔案
- 期間畫面黑黑的沒有任何反應，誤以為程式當機（Hang），於是按下 `Ctrl+C` 強制中止
- `Ctrl+C` 就會產生 `KeyboardInterrupt` 錯誤

**解決方法：**
- 執行指令後，把手離開鍵盤，**耐心等待**
- 等到畫面出現以下文字才代表成功啟動：
```
=======================================================
04_Inventory_Performance_Analysis — Raw Loader
=======================================================
```


---

### 📖 今日學習筆記

| 知識點 | 說明 |
|---|---|
| `python -u` | Unbuffered 模式，強制即時輸出 print 內容，不等暫存區滿才印出 |
| `DROP TABLE IF EXISTS` | PostgreSQL 找不到資料表時不報錯，只顯示 NOTICE |
| `load_dotenv()` | 從 `.env` 檔案讀取環境變數（資料庫密碼），避免密碼寫在程式碼中 |
| `os.getenv('KEY', 'fallback')` | 讀取環境變數，若找不到則使用預設值 |
| `encoding='utf-8-sig'` | 處理 CSV 的 BOM（\ufeff）問題，避免第一個欄位名稱被污染 |
| `dtype=str` | 讀取 CSV 時全部當作字串，避免 Pandas 自動型別推斷出錯 |
| 虛擬環境 (`04_env`) | 隔離專案套件，不影響電腦其他 Python 環境；用 `deactivate` 退出 |


---

### 🪲 踩坑紀錄 4：終端機關掉後，如何回看 Raw Loader 執行結果

**問題：**
- `01_load_raw.py` 執行完成後，終端機 / Console 視窗已經關掉
- 無法直接複製當時印出的 row count 給專案紀錄

**解法：**
- 本專案的 Python loader 已經設計了 `raw.load_audit` 稽核表
- 每次載入成功後，會把：
  - `table_name`
  - `rows_loaded`
  - `loaded_at`
  - `source_file`
  寫進 PostgreSQL

**回看載入結果的 SQL：**
```sql
SELECT 
    table_name, 
    rows_loaded, 
    loaded_at, 
    source_file 
FROM raw.load_audit
ORDER BY loaded_at DESC;
```

**用途：**
- 就算終端機關掉，也能從資料庫重新查回本次載入結果
- 適合拿來做：
  - row count 驗證
  - load 成功證明
  - README / 開發日誌補寫
  - 後續 staging 前的資料核對

**心得：**
- 長時間執行的資料載入流程，不要只依賴終端機畫面
- 最好把結果落地到 audit table，之後查詢會比找截圖更可靠



***

## Phase 2 驗證結果總覽

### ✅ 完全通過的檢查

| 檢查項目 | 結果 |
|---|---|
| `raw_sales` 負數 qty / dollars | **0** — 乾淨 |
| `raw_sales` price calc mismatch | **0** — SalesDollars = Qty × Price 完全吻合 |
| `raw_purchases` dollar calc mismatch | **0** — 採購金額計算正確 |
| `raw_beg/end_inventory` 負庫存 | **0** — 無異常 |
| Purchases vendor → Invoice vendor | **0** — 外鍵完整 |

***

### ⚠️ 需要處理的 3 個問題

**問題 1：`raw_purchase_prices` — price vs purchase_price 差異 12,260 筆** 

這是最重要的發現。兩欄並存且大量不一致，業務含義推測：
- `price` = **零售售價（Retail Price）**
- `purchase_price` = **進貨成本（Cost Price）**

這兩個欄位在 staging 層必須用明確的業務名稱區分，是計算 **Gross Margin** 和 **Inventory Turnover** 的關鍵。

**問題 2：`raw_purchase_prices` — price 欄位有 2 筆 zero/null** 

數量少，但需要在 staging 層用 `NULLIF` 處理，避免除零錯誤影響 DSI 計算。

**問題 3：Sales → Inventory 孤兒記錄大量存在** 
- Sales 中有 **64,044 個** `inventory_id` 在 `raw_beg_inventory` 找不到
- Sales 中有 **50,368 個** `inventory_id` 在 `raw_end_inventory` 找不到

這**不是錯誤**，是業務正常現象——有些商品在期初/期末已售罄，庫存表只記錄有在手庫存的品項。Staging 層用 `LEFT JOIN` 處理即可。

***

### 5a / 5b 無結果 = 好消息

**完全沒有重複記錄** ——`raw_sales` 和 `raw_purchase_prices` 的自然鍵均唯一，pgAdmin 顯示空結果是正確的行為（`HAVING COUNT(*) > 1` 沒有符合條件的行）。

***

## Phase 2 驗證結論

```
整體資料品質：良好 ✅
核心數值計算：完全準確 ✅
最大風險點：purchase_prices 雙 price 欄位語意不清 ⚠️
```









***

## 專案開發紀錄：Phase 2 ~ Phase 4 (Data Validation to Star Schema)
**日期**：2026-04-15
**階段目標**：將 1500 萬筆 Raw CSV 資料進行驗證、清洗、型別轉換，並最終建立適合 Power BI 分析的星型結構 (Star Schema)。

### 📍 Phase 2: 原始資料驗證 (Raw Data Validation)
**目標**：透過 SQL 檢驗載入 `raw` schema 的 6 張表，確認資料品質（Data Quality）、空值、業務邏輯與關聯完整性。


*   **業務邏輯發現**：
    *   `raw_purchase_prices` 表中同時存在 `price` 與 `purchase_price`，且有 12,260 筆不一致。經分析確立業務含義：`price` 為終端零售價 (Retail Price)，`purchase_price` 為進貨成本 (Cost Price)。
    *   部分 `vendor_name` 存在尾部空白字元（Trailing whitespace），需在 Staging 層清理。

### 📍 Phase 3: 暫存層轉換與清洗 (Staging Layer)
**目標**：建立 `stg_` 表，將所有 `TEXT` 型別轉換為正確的數值與日期型別，並統一命名規範與清理髒資料。

*   **問題 1：整數轉型失敗 (包含小數點的數量)**
    *   **狀況**：報錯 `invalid input syntax for type integer: "162.5"`。
    *   **解決**：真實世界的庫存數據（如秤重商品或容量）帶有小數。將 `quantity`, `sales_quantity`, `on_hand`, `volume` 等欄位從 `INTEGER` 改為 `NUMERIC(10,2)`。
*   **問題 2：字串混入數值欄位 (Dirty Data)**
    *   **狀況**：報錯 `invalid input syntax for type numeric: "Unknown"`。
    *   **解決**：在 1500 萬筆資料中混入了 `"Unknown"` 這樣的無效字串。實作了**防禦性轉型 (Defensive Casting)**，使用 `NULLIF(NULLIF(TRIM(column), ''), 'Unknown')` 將預期外的字串安全轉換為 `NULL`，確保 Data Pipeline 的穩健性。    
*   **根據驗證結果，Staging 層需要處理以下 4 件事：

1. **`raw_purchase_prices`** — 將 `price` 重命名為 `retail_price`，`purchase_price` 保留為 `purchase_price`，語意明確化
2. **所有 TEXT 欄位** — CAST 成正確類型（`NUMERIC`, `DATE`, `INTEGER`）
3. **VendorName TRIM** — 去除尾部空格（Section 7 已確認存在）
4. **`vendor_no` → `vendor_number`** — 統一 `raw_sales` 的命名與其他表一致

### 📍 Phase 4: 資料超市層建模 (Marts Layer - Star Schema)
**目標**：根據 Kimball 维度建模理論，建立事實表與維度表，並引入代理鍵 (Surrogate Keys, SK) 提升後續 BI 工具的關聯效能。

*   **建模產出**：
    *   **維度表 (Dimensions)**：`dim_product` (合併售價與成本基準), `dim_vendor`, `dim_store`。
    *   **事實表 (Facts)**：`fact_sales` (交易型), `fact_inventory_snapshot` (週期快照型，結合期初與期末)。
*   **問題 ：SQL 語法嚴格性 (Ambiguous Column & GROUP BY)**
    *   **狀況**：建立 `dim_product` 時，發生 `column reference "brand" is ambiguous` 及 `must appear in the GROUP BY clause` 錯誤。
    *   **解決**：PostgreSQL 對 Window Function `ROW_NUMBER() OVER(ORDER BY ...)` 內的表達式要求極高，必須與 `GROUP BY` 完全一致。透過明確指定別名 (Alias) 如 `COALESCE(p.brand, s.brand)` 並預先對子查詢進行聚合 (Pre-aggregation) 來優化 `FULL OUTER JOIN` 效能，最終成功建立維度表。

***

## Phase 5 



## 1. 資料連接策略：Import vs DirectQuery


**PostgreSQL 端的建議預處理（在 Power BI 連接前）：**


這個 `dim_date` 的意義在於：DAX 的 Time Intelligence 函數（`DATEADD`、`TOTALYTD` 等）**必須**依賴一張連續無缺漏的日期表，且該表須標記為 Date Table。直接從 `fact_sales` 的 `sales_date` 衍生無法保證連續性。

**Import 時的額外優化建議：**
- `fact_sales` 中的 `sales_date` 確保是 `DATE` 型別（非 timestamp），匯入後欄位會是整數型 date key，VertiPaq 壓縮率最高
- 如果記憶體有壓力，可在 PostgreSQL 建一個 View 只取 Power BI 需要的欄位，避免匯入不必要的欄位

***

## 2. Power BI 資料模型關聯設計

這是你的 Star Schema 在 Power BI 中的完整 Relationship 配置：

| From Table (Many) | From Column | To Table (One) | To Column | Cardinality | Direction |
|---|---|---|---|---|---|
| `fact_sales` | `product_sk` | `dim_product` | `product_sk` | Many-to-One | Single (→) |
| `fact_sales` | `store_sk` | `dim_store` | `store_sk` | Many-to-One | Single (→) |
| `fact_sales` | `vendor_sk` | `dim_vendor` | `vendor_sk` | Many-to-One | Single (→) |
| `fact_sales` | `sales_date` | `dim_date` | `date_key` | Many-to-One | Single (→) |
| `fact_inventory_snapshot` | `product_sk` | `dim_product` | `product_sk` | Many-to-One | Single (→) |
| `fact_inventory_snapshot` | `store_sk` | `dim_store` | `store_sk` | Many-to-One | Single (→) |
| `fact_inventory_snapshot` | `snapshot_date` | `dim_date` | `date_key` | Many-to-One | Single (→) |

**關鍵設計決策：**

- **兩張 Fact Table 都連接同一張 `dim_date`**，這正是你使用 `dim_date` 的核心理由——它是兩張事實表的 **Role-Playing Dimension** 橋梁
- **Cross-filter direction 全部設 Single（單向）**，避免 `dim_product` 的篩選意外透過 `fact_sales` 影響 `fact_inventory_snapshot`，這是標準 Star Schema 最佳實踐
- `dim_date` 建立後，在 Power BI 右鍵標記為 **"Mark as Date Table"**，否則 Time Intelligence DAX 函數不會生效

***

## 3. 核心 DAX — Inventory Turnover & DSI

你的 `fact_inventory_snapshot` 有 `BEGINNING` 與 `ENDING` 兩類快照，這是計算**平均庫存**的黃金資料，公式如下：


$$
\text{Inventory Turnover} 
= \frac{\text{COGS (期間銷售成本)}}{\text{Average Inventory}} 
$$


$$
    \text{DSI} = \frac{365}{\text{Inventory Turnover}} 
$$

### Step 1：基礎 Measure（建議建在獨立 `_Measures` 表中）



In [ ]:

-- 1. 期間總銷售成本（來自 fact_sales）
Total COGS =
SUMX(
    fact_sales,
    fact_sales[estimated_cogs]
)

-- 2. 期初庫存總值（BEGINNING 快照）
Beginning Inventory Value =
CALCULATE(
    SUM(fact_inventory_snapshot[total_inventory_value]),
    fact_inventory_snapshot[snapshot_type] = "BEGINNING"
)

-- 3. 期末庫存總值（ENDING 快照）
Ending Inventory Value =
CALCULATE(
    SUM(fact_inventory_snapshot[total_inventory_value]),
    fact_inventory_snapshot[snapshot_type] = "ENDING"
)

-- 4. 平均庫存（期初+期末 / 2）
Average Inventory Value =
DIVIDE(
    [Beginning Inventory Value] + [Ending Inventory Value],
    2
)



### Step 2：核心 KPI Measures


In [ ]:
-- 庫存周轉率 (Inventory Turnover)
Inventory Turnover =
DIVIDE(
    [Total COGS],
    [Average Inventory Value],
    BLANK()   -- 分母為 0 時回傳 BLANK 而非錯誤，避免 visual 顯示 Infinity
)

-- 庫存銷售天數 (Days Sales of Inventory)
-- 注意：分母用 Inventory Turnover，避免重複計算
Days Sales of Inventory (DSI) =
VAR _turnover = [Inventory Turnover]
RETURN
    IF(
        _turnover = 0 || ISBLANK(_turnover),
        BLANK(),
        DIVIDE(365, _turnover)
    )


### Step 3：Period-Aware 版本（支援 YTD 篩選）

當使用者在報表中選擇「年份」或「月份」時，上面的 Measures 已自動跟隨 `dim_date` 的篩選上下文正確計算。若你需要固定計算**全年度**對比，可加一個 YTD 版本：


In [ ]:
Inventory Turnover YTD =
CALCULATE(
    [Inventory Turnover],
    DATESYTD(dim_date[date_key])
)


***

## 

Phase 5 後續還需要的 DAX：
- **缺貨率**：需要用 `fact_inventory_snapshot` 中 `quantity_on_hand = 0` 的快照比例計算
- **呆滯庫存**：需定義「超過 N 天未銷售」的商品邏輯，建議用 `EXCEPT` 或 `DATESBETWEEN` 配合 `fact_sales` 做排除
- **ABC 分類**：用 `RANKX` 對品牌/商品排名，劃分 A/B/C 貢獻層

